# 3D reporter timelapse — 04_masked_illumination_qc

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Masked Illumination QC

This notebook reviews the masked illumination correction used before reporter quantification.

The goal is to show what the correction is doing, verify that it smooths position-dependent background differences, and keep the image evidence visible in the main pipeline rather than burying it in an archive stage.


## Setup

This section loads the saved masked-field estimate, the per-position diagnostics, and the helper functions used for image review.


### Load Analysis Packages


In [ ]:
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import Markdown, display
from skimage import morphology

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


### Load Saved Illumination QC Outputs


In [ ]:
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

DATASET_DIR = ROOT / "data/raw/20260128_BMP4-reporter_LPM-organoids/d2-d5"
ACQUISITION_SUMMARY_PATH = ROOT / "results/qc/acquisition_qc_summary.json"
FIELD_PATH = ROOT / "results/qc/04_masked_illumination_fields.npz"
DIAGNOSTICS_PATH = ROOT / "results/tables/04_masked_illumination_position_diagnostics.tsv"
PARAMETERS_PATH = ROOT / "results/qc/04_masked_illumination_parameters.json"

PREVIEW_DIR = ROOT / "results/previews/04_masked_illumination_qc"
FIGURE_DIR = ROOT / "results/figures/04"
TABLE_DIR = ROOT / "results/tables"
QC_DIR = ROOT / "results/qc"

PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

diagnostics = pd.read_csv(DIAGNOSTICS_PATH, sep="\t")
acquisition_summary = json.loads(ACQUISITION_SUMMARY_PATH.read_text())
parameters = json.loads(PARAMETERS_PATH.read_text())
field_payload = np.load(FIELD_PATH)

rfp_field = field_payload["rfp_field"].astype(np.float32)
yfp_field = field_payload["yfp_field"].astype(np.float32)
rfp_support = field_payload["rfp_sample_count"].astype(np.float32)
yfp_support = field_payload["yfp_sample_count"].astype(np.float32)

POSITION_RE = re.compile(r"Pos(?P<position_index>\d+)$")
REPORTER_CHANNELS = {"RFP": 1, "YFP": 2}
FIELD_BY_REPORTER = {"RFP": rfp_field, "YFP": yfp_field}
SUPPORT_BY_REPORTER = {"RFP": rfp_support, "YFP": yfp_support}
TIME_DISPLAY_OFFSET_HOURS = 48.0
INTERVAL_HOURS = float(acquisition_summary["interval_minutes"]) / 60.0

print("Project root:", ROOT)
print("Diagnostics rows:", len(diagnostics))
print("Sampled clean frames:", parameters["sampled_clean_frame_count"])
print("Used frames by channel:", parameters["used_frames_by_channel"])


### Define Helper Functions


In [ ]:
def position_index_from_label(position_label: str) -> int:
    match = POSITION_RE.fullmatch(position_label)
    if not match:
        raise ValueError(f"Unexpected position label: {position_label}")
    return int(match.group("position_index"))


def display_time_hours_from_index(time_index: int | float) -> float:
    return float(time_index) * INTERVAL_HOURS + TIME_DISPLAY_OFFSET_HOURS


def format_display_hours_from_index(time_index: int | float, decimals: int = 1) -> str:
    return f"{display_time_hours_from_index(time_index):.{decimals}f} h"


def raw_frame_path(position_label: str, channel_index: int, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        DATASET_DIR
        / position_label
        / f"img_channel{channel_index:03d}_position{position_index:03d}_time{time_index:09d}_z000.tif"
    )


def load_reporter(position_label: str, reporter: str, time_index: int) -> np.ndarray:
    return tiff.imread(raw_frame_path(position_label, REPORTER_CHANNELS[reporter], time_index)).astype(np.float32)


def load_mask(mask_path_str: str) -> np.ndarray:
    return tiff.imread(Path(mask_path_str)).astype(bool)


def annulus_mask(organoid_mask: np.ndarray, inner: int, outer: int, min_ring_pixels: int) -> np.ndarray:
    inner_mask = morphology.binary_dilation(organoid_mask, morphology.disk(inner))
    outer_mask = morphology.binary_dilation(organoid_mask, morphology.disk(outer))
    ring = outer_mask & ~inner_mask
    if int(ring.sum()) < min_ring_pixels:
        ring = ~outer_mask
    if int(ring.sum()) < min_ring_pixels:
        ring = ~organoid_mask
    return ring


def display_image(image: np.ndarray, low: float, high: float) -> np.ndarray:
    if not np.isfinite(low) or not np.isfinite(high) or math.isclose(high, low):
        high = low + 1.0
    return np.clip((image.astype(float) - low) / (high - low), 0, 1)


def pooled_limits(images: list[np.ndarray], q_low: float = 1.0, q_high: float = 99.5) -> tuple[float, float]:
    values = np.concatenate([img[np.isfinite(img)].ravel() for img in images if np.isfinite(img).any()])
    if values.size == 0:
        return 0.0, 1.0
    low, high = np.percentile(values, [q_low, q_high])
    if math.isclose(float(high), float(low)):
        high = low + 1.0
    return float(low), float(high)


def draw_contour(ax, mask: np.ndarray, color: str, linewidth: float = 1.5, linestyle: str = "solid") -> None:
    if mask.any():
        ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=linewidth, linestyles=linestyle)


def crop_bounds(cx: float, cy: float, image_shape: tuple[int, int], half_width: int = 72) -> tuple[slice, slice]:
    h, w = image_shape
    x0 = max(0, int(round(cx)) - half_width)
    x1 = min(w, int(round(cx)) + half_width)
    y0 = max(0, int(round(cy)) - half_width)
    y1 = min(h, int(round(cy)) + half_width)
    return slice(y0, y1), slice(x0, x1)


def corrected_reporter(position_label: str, reporter: str, time_index: int) -> np.ndarray:
    raw = load_reporter(position_label, reporter, time_index)
    field = FIELD_BY_REPORTER[reporter]
    return raw / np.clip(field, 1e-6, None)


## Review Estimated Illumination Fields

These panels show the smooth background-only field estimate and how much off-organoid support each pixel had during the estimation.


In [ ]:
centroid_df = diagnostics[["position_label", "centroid_x_px", "centroid_y_px"]].drop_duplicates().reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)
panels = [
    (axes[0, 0], rfp_field, "RFP masked illumination field", "magma"),
    (axes[0, 1], yfp_field, "YFP masked illumination field", "magma"),
    (axes[1, 0], rfp_support, "RFP support count", "viridis"),
    (axes[1, 1], yfp_support, "YFP support count", "viridis"),
]

for ax, image, title, cmap in panels:
    im = ax.imshow(image, cmap=cmap)
    if "field" in title.lower():
        ax.scatter(
            centroid_df["centroid_x_px"],
            centroid_df["centroid_y_px"],
            s=18,
            facecolors="none",
            edgecolors="white",
            linewidths=0.8,
            label="Diagnostic-frame cyst centroids",
        )
        ax.legend(
            loc="lower left",
            frameon=True,
            facecolor="black",
            edgecolor="white",
            framealpha=0.45,
            fontsize=8,
            labelcolor="white",
        )
        ax.set_title(title + "\nwhite circles = diagnostic-frame cyst centroids")
    else:
        ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

field_fig_path = FIGURE_DIR / "04_masked_illumination_fields.png"
fig.savefig(field_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote field figure:", field_fig_path)


## Review Local Annulus Diagnostics

The question here is simple: if a cyst sits in a brighter or dimmer part of the image field, does the local annulus background follow that field value less strongly after correction?


### One Concrete Example Before The Scatter Plots

This figure shows exactly what goes into one point on the upcoming annulus-vs-field scatter plots.

- The **field map** panel shows the smooth masked illumination field.
- The **highlighted centroid** marks the cyst location used to read the x-axis value.
- The **lime dashed annulus** around the cyst is the local off-organoid region used to read the y-axis value.
- The raw and corrected reporter panels let you see whether the correction changes the local background around the cyst in a sensible way.


In [ ]:
yfp_diag = diagnostics.loc[diagnostics["reporter"] == "YFP"].copy()
yfp_diag["distance_to_unity"] = (yfp_diag["field_at_centroid"] - 1.0).abs()
example_row = yfp_diag.sort_values(["distance_to_unity", "position_label"], ascending=[False, True]).iloc[0]
example_position = str(example_row["position_label"])
example_time = int(example_row["time_index"])
example_mask = load_mask(example_row["mask_path"])
example_ring = annulus_mask(
    example_mask,
    inner=int(parameters["background_inner"]),
    outer=int(parameters["background_outer"]),
    min_ring_pixels=int(parameters["min_ring_pixels"]),
)

example_raw = load_reporter(example_position, "YFP", example_time)
example_corrected = corrected_reporter(example_position, "YFP", example_time)
example_low, example_high = pooled_limits([example_raw, example_corrected], q_low=1.0, q_high=99.7)

example_fig, example_axes = plt.subplots(1, 3, figsize=(12.5, 4.2), constrained_layout=True)

field_ax = example_axes[0]
field_im = field_ax.imshow(yfp_field, cmap="magma")
field_ax.scatter(
    centroid_df["centroid_x_px"],
    centroid_df["centroid_y_px"],
    s=14,
    facecolors="none",
    edgecolors="white",
    linewidths=0.6,
    label="Diagnostic-frame cyst centroids",
)
field_ax.scatter(
    [float(example_row["centroid_x_px"])],
    [float(example_row["centroid_y_px"])],
    s=80,
    facecolors="none",
    edgecolors="cyan",
    linewidths=1.6,
    label=f"Highlighted example: {example_position}",
)
field_ax.set_title(
    f"YFP illumination field\ncyan = example centroid for {example_position}"
)
field_ax.legend(
    loc="lower left",
    frameon=True,
    facecolor="black",
    edgecolor="white",
    framealpha=0.45,
    fontsize=7,
    labelcolor="white",
)
field_ax.axis("off")
example_fig.colorbar(field_im, ax=field_ax, fraction=0.046, pad=0.02)

raw_ax = example_axes[1]
raw_ax.imshow(display_image(example_raw, example_low, example_high), cmap="gray")
draw_contour(raw_ax, example_mask, color="deepskyblue", linewidth=1.5)
draw_contour(raw_ax, example_ring, color="lime", linewidth=1.0, linestyle="dashed")
raw_ax.set_title(
    f"YFP raw, {example_position} t={format_display_hours_from_index(example_time, 0)}\n"
    f"annulus median = {float(example_row['raw_annulus_median']):.1f}"
)
raw_ax.axis("off")

corr_ax = example_axes[2]
corr_ax.imshow(display_image(example_corrected, example_low, example_high), cmap="gray")
draw_contour(corr_ax, example_mask, color="deepskyblue", linewidth=1.5)
draw_contour(corr_ax, example_ring, color="lime", linewidth=1.0, linestyle="dashed")
corr_ax.set_title(
    f"YFP corrected, {example_position} t={format_display_hours_from_index(example_time, 0)}\n"
    f"annulus median = {float(example_row['corrected_annulus_median']):.1f}"
)
corr_ax.axis("off")

example_fig_path = FIGURE_DIR / "04_masked_illumination_annulus_field_example.png"
example_fig.savefig(example_fig_path, dpi=180, bbox_inches="tight")
display(example_fig)
plt.close(example_fig)

display(
    Markdown(
        f'''
        ### How This Maps To One Scatter-Plot Point

        - **x-axis value** for this example comes from the field value at the cyan-highlighted centroid: `{float(example_row["field_at_centroid"]):.3f}`
        - **y-axis raw value** comes from the median intensity in the lime dashed annulus on the raw frame: `{float(example_row["raw_annulus_median"]):.1f}`
        - **y-axis corrected value** comes from that same annulus after masked-field correction: `{float(example_row["corrected_annulus_median"]):.1f}`
        '''
    )
)
print("Wrote annulus-field example figure:", example_fig_path)

corr_rows = []
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True, sharex="col")
for row_index, reporter in enumerate(["RFP", "YFP"]):
    sub = diagnostics.loc[diagnostics["reporter"] == reporter].copy()
    raw_r = float(np.corrcoef(sub["field_at_centroid"], sub["raw_annulus_median"])[0, 1])
    corr_r = float(np.corrcoef(sub["field_at_centroid"], sub["corrected_annulus_median"])[0, 1])
    corr_rows.append(
        {
            "reporter": reporter,
            "raw_corr_vs_field": raw_r,
            "corrected_corr_vs_field": corr_r,
            "raw_cv": float(np.nanstd(sub["raw_annulus_median"]) / np.nanmean(sub["raw_annulus_median"])),
            "corrected_cv": float(np.nanstd(sub["corrected_annulus_median"]) / np.nanmean(sub["corrected_annulus_median"])),
        }
    )

    axes[row_index, 0].scatter(sub["field_at_centroid"], sub["raw_annulus_median"], color="tab:blue", alpha=0.8)
    axes[row_index, 0].set_title(f"{reporter} raw annulus vs field (r={raw_r:.3f})")
    axes[row_index, 0].set_ylabel("Raw annulus median")

    axes[row_index, 1].scatter(sub["field_at_centroid"], sub["corrected_annulus_median"], color="tab:green", alpha=0.8)
    axes[row_index, 1].set_title(f"{reporter} corrected annulus vs field (r={corr_r:.3f})")

axes[1, 0].set_xlabel("Field value at cyst centroid")
axes[1, 1].set_xlabel("Field value at cyst centroid")

annulus_fig_path = FIGURE_DIR / "04_masked_illumination_annulus_scatter.png"
fig.savefig(annulus_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

annulus_summary = pd.DataFrame(corr_rows)
display(
    Markdown(
        "\n".join(
            [
                "### Plain-Language Read",
                *[
                    (
                        f"- **{row.reporter}**: correlation of annulus median with field value "
                        f"drops from `{row.raw_corr_vs_field:.3f}` to `{row.corrected_corr_vs_field:.3f}`; "
                        f"CV changes from `{row.raw_cv:.4f}` to `{row.corrected_cv:.4f}`."
                    )
                    for row in annulus_summary.itertuples(index=False)
                ],
            ]
        )
    )
)
print("Wrote annulus diagnostic figure:", annulus_fig_path)


## Select Representative Positions For Image Review

The goal here is not to show every position. Instead, the notebook picks a small set spanning low-field, near-unity, and high-field YFP locations so that the correction can be judged by eye where it should matter most.


In [ ]:
yfp_diag = diagnostics.loc[diagnostics["reporter"] == "YFP"].copy().sort_values("field_at_centroid").reset_index(drop=True)

low_positions = yfp_diag.head(3)["position_label"].tolist()
high_positions = yfp_diag.tail(3)["position_label"].tolist()
remaining = yfp_diag.loc[~yfp_diag["position_label"].isin(low_positions + high_positions)].copy()
remaining["distance_to_unity"] = (remaining["field_at_centroid"] - 1.0).abs()
middle_positions = remaining.nsmallest(3, "distance_to_unity")["position_label"].tolist()

review_positions = low_positions + middle_positions + high_positions

review_df = yfp_diag.loc[yfp_diag["position_label"].isin(review_positions)].copy()
review_df["group"] = np.where(
    review_df["position_label"].isin(low_positions),
    "low_field",
    np.where(review_df["position_label"].isin(high_positions), "high_field", "near_unity"),
)
review_df["group_order"] = review_df["group"].map({"low_field": 0, "near_unity": 1, "high_field": 2})
review_df = review_df.sort_values(["group_order", "field_at_centroid", "position_label"]).reset_index(drop=True)
review_positions = review_df["position_label"].tolist()

review_positions_path = TABLE_DIR / "04_masked_illumination_review_positions.tsv"
review_df.assign(
    time_hours=review_df["time_index"].map(display_time_hours_from_index)
)[["position_label", "time_index", "time_hours", "field_at_centroid", "group"]].to_csv(review_positions_path, sep="\t", index=False)

display(
    Markdown(
        f'''
        ### Review Positions

        `{", ".join(review_positions)}`

        Groups:
        - low field: `{", ".join(low_positions)}`
        - near unity: `{", ".join(middle_positions)}`
        - high field: `{", ".join(high_positions)}`
        '''
    )
)
print("Wrote review-position list:", review_positions_path)


## Review Whole-Frame Before And After Images

These panels use one early clean frame per selected position, with the cyst mask in cyan and the local annulus in lime. Raw and corrected images use a shared reporter-specific display scale.


In [ ]:
frame_lookup = {
    (str(row.position_label), str(row.reporter)): int(row.time_index)
    for row in diagnostics.itertuples(index=False)
}

raw_corrected_cache = {}
for reporter in ["RFP", "YFP"]:
    for position_label in review_positions:
        time_index = frame_lookup[(position_label, reporter)]
        raw = load_reporter(position_label, reporter, time_index)
        corrected = corrected_reporter(position_label, reporter, time_index)
        raw_corrected_cache[(position_label, reporter, "raw")] = raw
        raw_corrected_cache[(position_label, reporter, "corrected")] = corrected

whole_limits = {}
for reporter in ["RFP", "YFP"]:
    images = []
    for position_label in review_positions:
        images.append(raw_corrected_cache[(position_label, reporter, "raw")])
        images.append(raw_corrected_cache[(position_label, reporter, "corrected")])
    whole_limits[reporter] = pooled_limits(images, q_low=1.0, q_high=99.7)

for reporter in ["RFP", "YFP"]:
    fig, axes = plt.subplots(len(review_positions), 2, figsize=(8.8, 2.8 * len(review_positions)), constrained_layout=True)
    if len(review_positions) == 1:
        axes = np.expand_dims(axes, axis=0)

    low, high = whole_limits[reporter]
    reporter_diag = diagnostics.loc[diagnostics["reporter"] == reporter].set_index("position_label")

    for row_index, position_label in enumerate(review_positions):
        time_index = frame_lookup[(position_label, reporter)]
        diag_row = reporter_diag.loc[position_label]
        mask = load_mask(diag_row["mask_path"])
        ring = annulus_mask(
            mask,
            inner=int(parameters["background_inner"]),
            outer=int(parameters["background_outer"]),
            min_ring_pixels=int(parameters["min_ring_pixels"]),
        )

        for col_index, mode in enumerate(["raw", "corrected"]):
            image = raw_corrected_cache[(position_label, reporter, mode)]
            ax = axes[row_index, col_index]
            ax.imshow(display_image(image, low, high), cmap="gray")
            draw_contour(ax, mask, color="deepskyblue", linewidth=1.5)
            draw_contour(ax, ring, color="lime", linewidth=1.0, linestyle="dashed")
            if row_index == 0:
                ax.set_title(f"{reporter} {mode}")
            if col_index == 0:
                ax.set_ylabel(
                    f"{position_label}\nt={format_display_hours_from_index(time_index, 0)}\nfield={diag_row['field_at_centroid']:.3f}"
                )
            annulus_value = diag_row[f"{mode}_annulus_median"]
            ax.text(
                0.02,
                0.98,
                f"annulus={annulus_value:.1f}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                color="white",
                fontsize=8,
                bbox={"facecolor": "black", "alpha": 0.45, "pad": 2},
            )
            ax.axis("off")

    whole_path = FIGURE_DIR / f"04_masked_illumination_{reporter.lower()}_wholeframe_review.png"
    fig.savefig(whole_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote whole-frame review:", whole_path)


## Review Local Cyst Crops

These crop panels make it easier to see whether the correction changes the local reporter neighborhood around the organoid in a plausible way, rather than just changing the empty background.


In [ ]:
crop_limits = {}
for reporter in ["RFP", "YFP"]:
    crops = []
    reporter_diag = diagnostics.loc[diagnostics["reporter"] == reporter].set_index("position_label")
    for position_label in review_positions:
        diag_row = reporter_diag.loc[position_label]
        mask = load_mask(diag_row["mask_path"])
        ys, xs = crop_bounds(
            cx=float(diag_row["centroid_x_px"]),
            cy=float(diag_row["centroid_y_px"]),
            image_shape=mask.shape,
            half_width=72,
        )
        crops.append(raw_corrected_cache[(position_label, reporter, "raw")][ys, xs])
        crops.append(raw_corrected_cache[(position_label, reporter, "corrected")][ys, xs])
    crop_limits[reporter] = pooled_limits(crops, q_low=1.0, q_high=99.7)

for reporter in ["RFP", "YFP"]:
    fig, axes = plt.subplots(len(review_positions), 2, figsize=(8.8, 2.8 * len(review_positions)), constrained_layout=True)
    if len(review_positions) == 1:
        axes = np.expand_dims(axes, axis=0)

    low, high = crop_limits[reporter]
    reporter_diag = diagnostics.loc[diagnostics["reporter"] == reporter].set_index("position_label")

    for row_index, position_label in enumerate(review_positions):
        diag_row = reporter_diag.loc[position_label]
        mask = load_mask(diag_row["mask_path"])
        ring = annulus_mask(
            mask,
            inner=int(parameters["background_inner"]),
            outer=int(parameters["background_outer"]),
            min_ring_pixels=int(parameters["min_ring_pixels"]),
        )
        ys, xs = crop_bounds(
            cx=float(diag_row["centroid_x_px"]),
            cy=float(diag_row["centroid_y_px"]),
            image_shape=mask.shape,
            half_width=72,
        )
        mask_crop = mask[ys, xs]
        ring_crop = ring[ys, xs]

        for col_index, mode in enumerate(["raw", "corrected"]):
            crop = raw_corrected_cache[(position_label, reporter, mode)][ys, xs]
            ax = axes[row_index, col_index]
            ax.imshow(display_image(crop, low, high), cmap="gray")
            draw_contour(ax, mask_crop, color="deepskyblue", linewidth=1.5)
            draw_contour(ax, ring_crop, color="lime", linewidth=1.0, linestyle="dashed")
            if row_index == 0:
                ax.set_title(f"{reporter} {mode} crop")
            if col_index == 0:
                ax.set_ylabel(position_label)
            ax.axis("off")

    crop_path = FIGURE_DIR / f"04_masked_illumination_{reporter.lower()}_crop_review.png"
    fig.savefig(crop_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote crop review:", crop_path)


## Interpretation

The notebook should answer three practical questions:

- Is the estimated field smooth and plausible, or is it obviously learning organoid structure?
- Does the correction weaken the link between local annulus background and field position?
- In the representative image panels, do the corrected frames look more comparable across positions without flattening genuine cyst-local fluorescence structure?


In [ ]:
summary_lines = ["### Current Prototype Read"]
for row in annulus_summary.itertuples(index=False):
    trend = "weaker" if abs(row.corrected_corr_vs_field) < abs(row.raw_corr_vs_field) else "not improved"
    summary_lines.append(
        (
            f"- **{row.reporter}**: annulus dependence on field position becomes {trend} "
            f"(`r {row.raw_corr_vs_field:.3f} -> {row.corrected_corr_vs_field:.3f}`), "
            f"with a small CV change (`{row.raw_cv:.4f} -> {row.corrected_cv:.4f}`)."
        )
    )

summary_lines.extend(
    [
        "- The image panels matter more than the scatter alone. If the correction makes the background more comparable but starts to distort real cyst-local structure, it should not be adopted.",
        "- This correction uses only off-organoid pixels, precisely because a naive median-combined flat-field would be contaminated by persistent central cyst occupancy in this dataset.",
    ]
)
display(Markdown("\n".join(summary_lines)))


## Save Stage Outputs


In [ ]:
notebook_summary = {
    "field_figure": str(FIGURE_DIR / "04_masked_illumination_fields.png"),
    "annulus_field_example_figure": str(FIGURE_DIR / "04_masked_illumination_annulus_field_example.png"),
    "annulus_scatter_figure": str(FIGURE_DIR / "04_masked_illumination_annulus_scatter.png"),
    "rfp_wholeframe_review": str(FIGURE_DIR / "04_masked_illumination_rfp_wholeframe_review.png"),
    "yfp_wholeframe_review": str(FIGURE_DIR / "04_masked_illumination_yfp_wholeframe_review.png"),
    "rfp_crop_review": str(FIGURE_DIR / "04_masked_illumination_rfp_crop_review.png"),
    "yfp_crop_review": str(FIGURE_DIR / "04_masked_illumination_yfp_crop_review.png"),
    "review_positions_tsv": str(TABLE_DIR / "04_masked_illumination_review_positions.tsv"),
    "field_npz": str(FIELD_PATH),
    "diagnostics_tsv": str(DIAGNOSTICS_PATH),
    "parameters_json": str(PARAMETERS_PATH),
}
summary_path = QC_DIR / "04_masked_illumination_notebook_outputs.json"
summary_path.write_text(json.dumps(notebook_summary, indent=2))
print("Wrote notebook output manifest:", summary_path)
display(pd.Series(notebook_summary, name="path").to_frame())
